# Material de apoyo — Tarea 3: Score Matching

Este notebook importa y ejecuta pruebas breves de la funcionalidad principal
de cada script del repositorio: `model.py`, `model_clf.py`, `train.py`,
`train_clf.py` y `sample.py`.

**Estructura del notebook:**
1. Configuración e imports
2. `model.py` — red de difusión (UNet condicional)
3. `model_clf.py` — clasificador auxiliar
4. `train.py` — entrenamiento de una red de difusión
5. `train_clf.py` — entrenamiento del clasificador auxiliar
6. `sample.py` — muestreo (SDE reversa con CFG)
7. (Opcional) Carga de checkpoints reales, si existen en `outputs/`

Las pruebas de las secciones 2–6 usan **datos sintéticos** (tensores
aleatorios) en vez de MNIST real, precisamente para que el notebook corra
rápido y sin depender de que el dataset ya esté descargado — solo se
verifica que la interfaz de cada función/clase funciona como se espera
(formas de tensores, valores retornados, etc.), no la calidad del
entrenamiento. La sección 7 sí usa los checkpoints reales si están
disponibles.


## 1. Configuración e imports

Se asume que este notebook vive en la **raíz del proyecto**, junto a
`model.py`, `model_clf.py`, `train.py`, `train_clf.py` y `sample.py`.

In [1]:
import sys
from pathlib import Path

import torch
from torch.utils.data import TensorDataset

# Aseguramos que la raíz del proyecto esté en el path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
print(f"Raíz del proyecto: {PROJECT_ROOT}")

Dispositivo: cuda
Raíz del proyecto: k:\Repositorios\Tarea_Topicos\Score-Matching


## 2. `model.py` — Red de difusión

Se instancia `Modelo` con la misma configuración usada en `train.py`
(`in_channels`, `out_channels`, `base_channels`, `embedding_dim`,
`n_labels`, `cfg`) y se prueban sus métodos principales:

- `forward(x, t, y)` → predicción de ruido $\epsilon_\theta(x_t, t, y)$
- `score(x, t, y)` → score $s_\theta = -\epsilon_\theta/\beta_t$
- `drift(score, x, t)` → drift de la SDE reversa
- `alpha(t)`, `beta(t)` → coeficientes del camino gaussiano
- `token_nulo` → índice reservado para $\varnothing$ (solo relevante si `cfg=True`)

Todo se prueba con tensores aleatorios de forma `(B, 1, 16, 16)`, que es la
resolución real usada en el proyecto (MNIST redimensionado a 16×16).

In [2]:
from model import Modelo

config_difusion = {
    "in_channels": 1,
    "out_channels": 1,
    "base_channels": 32,
    "embedding_dim": 256,
    "n_labels": 10,
    "cfg": True,  # red con CFG (incluye token nulo)
}

modelo_test = Modelo(**config_difusion).to(DEVICE)

# Batch de prueba
x = torch.randn(4, 1, 16, 16, device=DEVICE)
t = torch.rand(4, device=DEVICE)
y = torch.randint(0, 10, (4,), device=DEVICE)

eps_pred = modelo_test(x, t, y)
score = modelo_test.score(x, t, y)
drift = modelo_test.drift(score, x, t)

print("forward(x,t,y)  ->", eps_pred.shape)
print("score(x,t,y)    ->", score.shape)
print("drift(s,x,t)    ->", drift.shape)
print("alpha(t)        ->", modelo_test.alpha(t).shape)
print("beta(t)         ->", modelo_test.beta(t).shape)
print("token_nulo      ->", modelo_test.token_nulo)

assert eps_pred.shape == x.shape
assert score.shape == x.shape
assert drift.shape == x.shape
print("\nOK: todas las formas de salida coinciden con la forma de entrada.")

forward(x,t,y)  -> torch.Size([4, 1, 16, 16])
score(x,t,y)    -> torch.Size([4, 1, 16, 16])
drift(s,x,t)    -> torch.Size([4, 1, 16, 16])
alpha(t)        -> torch.Size([4])
beta(t)         -> torch.Size([4])
token_nulo      -> 10

OK: todas las formas de salida coinciden con la forma de entrada.


## 3. `model_clf.py` — Clasificador auxiliar

Se instancia `Clasificador(**config)` y se verifica que `forward(x)` retorna
logits crudos de forma `(B, num_clases)`, sin depender de `t` ni de `y`
(a diferencia de `Modelo`, ya que el clasificador resuelve un problema de
clasificación cerrado sobre datos limpios).

In [3]:
from model_clf import Clasificador

config_clf = {"num_clases": 10, "dropout": 0.2}
clf_test = Clasificador(**config_clf).to(DEVICE)

x_img = torch.randn(4, 1, 16, 16, device=DEVICE)
logits = clf_test(x_img)

print("forward(x) -> logits:", logits.shape)
assert logits.shape == (4, 10)
print("OK: logits de forma (B, num_clases), sin Softmax aplicado.")
print("\nEjemplo de logits crudos (primera imagen):\n", logits[0].detach().cpu())

forward(x) -> logits: torch.Size([4, 10])
OK: logits de forma (B, num_clases), sin Softmax aplicado.

Ejemplo de logits crudos (primera imagen):
 tensor([ 0.2973,  0.0686,  0.1545, -0.1190, -0.1501,  0.3219,  0.0496, -0.2846,
        -0.1331,  0.3623])


## 4. `train.py` — Entrenamiento de la red de difusión

Se prueba `train()` con un **dataset sintético** (tensores aleatorios) por
una sola época, para verificar que el ciclo completo —forward, pérdida DSM
(`MSELoss` entre ruido real y predicho), backward, optimizer step, y
`label_dropout`— corre sin errores.

> Nota: con datos sintéticos la pérdida no tiene por qué bajar de forma
> significativa; el objetivo aquí es solo confirmar que la interfaz
> funciona, no evaluar convergencia.

In [4]:
import train as train_mod

# Dataset sintético: imágenes aleatorias + etiquetas aleatorias, mismo shape que MNIST 16x16
N = 32
X_dummy = torch.randn(N, 1, 16, 16)
y_dummy = torch.randint(0, 10, (N,))
dataset_dummy = TensorDataset(X_dummy, y_dummy)

historial_dummy = train_mod.train(
    model=modelo_test,
    data=dataset_dummy,
    n_epochs=1,
    batch_size=8,
    lr=1e-3,
    device=DEVICE,
    seed=42,
    label_dropout=0.3,
)

print("Historial de pérdida (1 época, datos sintéticos):", historial_dummy)
assert len(historial_dummy) == 1
print("OK: train() ejecuta el ciclo completo (forward, pérdida DSM, backward, optimizer.step) sin errores.")

Entrenando modelo: 100%|██████████| 1/1 [00:03<00:00,  3.82s/it]


Historial de pérdida (1 época, datos sintéticos): [0.9327009171247482]
OK: train() ejecuta el ciclo completo (forward, pérdida DSM, backward, optimizer.step) sin errores.


## 5. `train_clf.py` — Entrenamiento del clasificador auxiliar

Se prueba `train()` de forma análoga, con datasets sintéticos de
entrenamiento y validación. A diferencia de `train.py`, esta función además
guarda un checkpoint de reanudación (`checkpoint_path`) tras cada época —se
usa un archivo temporal para no sobrescribir un `clasificador.pt` real.

In [5]:
import train_clf as train_clf_mod

N_train, N_val = 32, 8
X_train_dummy = torch.randn(N_train, 1, 16, 16)
y_train_dummy = torch.randint(0, 10, (N_train,))
X_val_dummy = torch.randn(N_val, 1, 16, 16)
y_val_dummy = torch.randint(0, 10, (N_val,))

train_ds_dummy = TensorDataset(X_train_dummy, y_train_dummy)
val_ds_dummy = TensorDataset(X_val_dummy, y_val_dummy)

ckpt_test_path = Path("clasificador_test_tmp.pt")
if ckpt_test_path.exists():
    ckpt_test_path.unlink()  # limpiar corridas de prueba anteriores

historial_clf_dummy = train_clf_mod.train(
    model=clf_test,
    data=train_ds_dummy,
    n_epochs=1,
    batch_size=8,
    lr=1e-3,
    device=DEVICE,
    seed=42,
    checkpoint_path=ckpt_test_path,
    val_data=val_ds_dummy,
)

print("Historial (1 época, datos sintéticos):", historial_clf_dummy)
assert set(historial_clf_dummy.keys()) == {"train_loss", "train_acc", "val_loss", "val_acc"}
print("OK: train() corre el ciclo train/val y guarda checkpoint de reanudación.")

# Limpieza del checkpoint de prueba
if ckpt_test_path.exists():
    ckpt_test_path.unlink()

Entrenando en cuda desde la época 1 hasta 1...


RuntimeError: DataLoader worker (pid(s) 21240, 19652, 18828, 19332, 16308, 21416, 3368, 17112) exited unexpectedly

## 6. `sample.py` — Muestreo (SDE reversa con CFG)

Se prueba `sample()` integrando la SDE reversa con pocos pasos (`n_steps=5`,
en vez de los 250 usados en producción) sobre el modelo de prueba recién
instanciado, y se verifica que:

- las imágenes generadas tienen la forma esperada `(B, C, H, W)`,
- `imgs_per_step` contiene las capturas intermedias del proceso,
- las funciones de exportación (`exportar_grilla_muestras`,
  `exportar_proceso_muestreo`) generan archivos `.png` sin errores.

In [6]:
import sample as sample_mod

sigma_fn = lambda t: 0.5 * torch.ones_like(t)  # mismo sigma constante usado en sample.py

ruido_inicial = torch.randn(4, 1, 16, 16, device=DEVICE)
labels_sample = torch.tensor([1, 2, 3, 4], device=DEVICE)

samples, proceso = sample_mod.sample(
    modelo_test,
    ruido_inicial,
    n_steps=5,       # pocos pasos solo para la prueba (producción usa 250)
    w=1.5,
    device=DEVICE,
    sigma=sigma_fn,
    labels=labels_sample,
)

print("samples.shape:", samples.shape)
print("Pasos capturados en imgs_per_step:", len(proceso))
assert samples.shape == ruido_inicial.shape
print("OK: sample() integra la SDE reversa y retorna imágenes con la forma esperada.")

samples.shape: torch.Size([4, 1, 16, 16])
Pasos capturados en imgs_per_step: 5
OK: sample() integra la SDE reversa y retorna imágenes con la forma esperada.


In [7]:
test_out_dir = Path("outputs_test")
test_out_dir.mkdir(exist_ok=True)

sample_mod.exportar_grilla_muestras(
    samples.detach().cpu(), labels_sample.cpu(),
    test_out_dir / "grilla_prueba.png", title="prueba",
)
sample_mod.exportar_proceso_muestreo(
    proceso, labels_sample.cpu(),
    test_out_dir / "proceso_prueba.png", title="prueba",
)

print("OK: grilla_prueba.png y proceso_prueba.png generados en", test_out_dir)

OK: grilla_prueba.png y proceso_prueba.png generados en outputs_test


## 7. (Opcional) Carga de checkpoints reales

Si ya entrenaste los modelos y tienes `modelo_cond.pt`, `modelo_cfg.pt` y
`clasificador.pt` en la carpeta `outputs/`, esta celda los carga y genera
una muestra real por clase, clasificándola con el clasificador auxiliar.
Si los checkpoints no existen todavía, la celda lo indica sin fallar.

In [8]:
OUTPUT_PATH = Path("outputs")
rutas = {
    "modelo_cond": OUTPUT_PATH / "modelo_cond.pt",
    "modelo_cfg": OUTPUT_PATH / "modelo_cfg.pt",
    "clasificador": OUTPUT_PATH / "clasificador.pt",
}

disponibles = {k: v.exists() for k, v in rutas.items()}
print("Checkpoints encontrados:", disponibles)

if all(disponibles.values()):
    # Cargar clasificador real
    ckpt_clf = torch.load(rutas["clasificador"], map_location=DEVICE, weights_only=False)
    clf_real = Clasificador(**ckpt_clf["model_config"]).to(DEVICE)
    clf_real.load_state_dict(ckpt_clf["model_state"])
    clf_real.eval()

    # Cargar red condicional pura (w=1)
    ckpt_cond = torch.load(rutas["modelo_cond"], map_location=DEVICE, weights_only=False)
    modelo_cond = Modelo(**ckpt_cond["model_config"]).to(DEVICE)
    modelo_cond.load_state_dict(ckpt_cond["model_state"])

    # Generamos 1 muestra por clase y las clasificamos con el juez auxiliar
    torch.manual_seed(42)
    ruido = torch.randn(10, 1, 16, 16, device=DEVICE)
    labels_reales = torch.arange(10, device=DEVICE)

    muestras, _ = sample_mod.sample(
        modelo_cond, ruido, n_steps=250, w=1.0, device=DEVICE,
        sigma=sigma_fn, labels=labels_reales,
    )

    with torch.no_grad():
        preds = clf_real(muestras).argmax(dim=1)

    fidelidad = (preds == labels_reales).float().mean().item()
    print(f"Fidelidad (modelo condicional, w=1) sobre 1 muestra/clase: {fidelidad*100:.1f}%")
    print("Etiquetas pedidas: ", labels_reales.cpu().tolist())
    print("Predicciones juez: ", preds.cpu().tolist())
else:
    faltantes = [k for k, v in disponibles.items() if not v]
    print(f"Faltan checkpoints: {faltantes}. Entrena los modelos primero (train.py / train_clf.py) "
          f"para poder ejecutar esta celda.")

Checkpoints encontrados: {'modelo_cond': True, 'modelo_cfg': True, 'clasificador': True}
Fidelidad (modelo condicional, w=1) sobre 1 muestra/clase: 70.0%
Etiquetas pedidas:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Predicciones juez:  [6, 1, 2, 2, 4, 5, 6, 7, 8, 7]
